# 02 · Behavioural gate — **HARD**

Attribution against a trait that never transferred is meaningless. This notebook
decides which A fraction, if any, carries a behaviourally visible trait, and
writes that choice to `RUN/gate.json` for every stage downstream.

**Selection rule, fixed before looking at the numbers:**

> A fraction *f* passes iff its 95% CI on the **substring** rate under the
> **plain** variant lies entirely above the clean student's:
> `ci_low(mix_f) > ci_high(clean)`.
> The chosen fraction is the **lowest** passing one.

Lowest, not best: the interesting regime is the weakest dose that still
transmits, because that is where an auditor would actually be working. Two
sanity conditions accompany it — `pureA ≫ clean` (the ceiling exists) and
`clean ≈ base` (the neutral corpus moves nothing).

If no fraction passes, this raises. That is the pre-registered stop.

In [ ]:
# --- bootstrap: identical first cell in every pivot notebook -----------------
# /workspace is the Runpod network volume, so `runs/` (which config.py resolves
# relative to the repo root) survives a pod stop. Nothing here writes to the
# container disk except the HF cache, which is redirected for the same reason.
import os, sys, json, time, hashlib, subprocess
from pathlib import Path

ROOT = Path("/workspace/subliminal-attrib")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("SUBATTR_THIRD_PARTY", str(ROOT / "third_party"))
os.environ.setdefault("HF_HOME", "/workspace/hf_home")
os.environ.setdefault("WANDB_MODE", "disabled")

%load_ext autoreload
%autoreload 2

import torch
from subattr import config

cfg  = config.load("configs/pivot.yaml")
DATA = cfg.data_dir
RUN  = cfg.run_dir
MIX  = DATA / "mixtures"
T0   = time.time()

# WHICH code is running? Nothing else in this notebook would notice a `main`
# checkout until a missing file several cells in, and pivot and main answer
# different questions -- their results have to stay independently attributable.
# sys.path puts ROOT/src first so the working tree beats any installed copy;
# assert that rather than assume it.
BRANCH = subprocess.run(
    ["git", "-C", str(ROOT), "rev-parse", "--abbrev-ref", "HEAD"],
    capture_output=True, text=True,
).stdout.strip()
assert BRANCH == "pivot", f"expected the 'pivot' branch at {ROOT}, found {BRANCH!r}"
assert Path(config.__file__).resolve().is_relative_to(ROOT / "src"), (
    f"subattr is imported from {config.__file__}, not {ROOT / 'src'}"
)
assert config.REPO_ROOT == ROOT, f"REPO_ROOT is {config.REPO_ROOT}, not {ROOT}"

print(f"config    {cfg.name}   model_hash={cfg.hash}   data_hash={cfg.data_hash}")
print(f"branch    {BRANCH}   {config.git_sha()}")
print(f"gpu       {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
print(f"data_dir  {DATA}")
print(f"run_dir   {RUN}")

In [ ]:
from subattr import behavior as bh
from subattr import train as tr

FRACTIONS = ("mix10", "mix25", "mix50")
ARMS = ("clean",) + FRACTIONS + ("pureA",)
adapters = {name: tr.latest_adapter(str(RUN / "students" / name)) for name in ARMS}
for name, path in adapters.items():
    print(f"  {name:<8s} {path}")

In [ ]:
# ~40 min on an H100: 6 models x 2 variants x 50 prompts x 50 samples.
# Cached, so a dropped session costs the load and not the generation.
results = bh.probe_adapters(
    cfg.base_model,
    adapters,
    target_word=cfg.entity_a,
    variants=("plain", "numbers_prefix"),
    include_base=True,
    cache_path=str(RUN / "behavior.json"),
)
print(f"{len(results)} results")

In [ ]:
by = {(r.label, r.variant): r for r in results}
order = ("base",) + ARMS
print(f"{'arm':<10s} {'variant':<15s} {'P(cat) substring':>18s}   95% CI")
for name in order:
    for variant in ("plain", "numbers_prefix"):
        r = by[(name, variant)]
        print(f"{name:<10s} {variant:<15s} {r.rate_substring:>18.4f}   "
              f"[{r.ci_low_substring:.4f}, {r.ci_high_substring:.4f}]")

print()
for name in FRACTIONS + ("pureA",):
    print("  " + bh.paired_difference(by[(name, "plain")], by[("clean", "plain")]).line())

In [ ]:
RULE = "ci_low(mix_f) > ci_high(clean) on the plain-variant substring rate; lowest passing f"

clean_plain = by[("clean", "plain")]
passing = [
    f for f in FRACTIONS
    if by[(f, "plain")].ci_low_substring > clean_plain.ci_high_substring
]
print(f"rule    : {RULE}")
print(f"clean   : {clean_plain.rate_substring:.4f} "
      f"[{clean_plain.ci_low_substring:.4f}, {clean_plain.ci_high_substring:.4f}]")
print(f"passing : {passing or 'NONE'}")

if not passing:
    raise SystemExit(
        "GATE FAILED: no A fraction is behaviourally separable from the clean student.\n"
        "This is a result, not a bug -- PLAN v2's pivot is to report the dose-response "
        "curve itself (rate vs fraction, with CIs, against the pureA ceiling) and to "
        "state that attribution was not attempted because there was no transferred "
        "trait to attribute. Do NOT proceed to 04-09."
    )

In [ ]:
# Sanity conditions on the two ends of the dose axis.
base_plain, pure_plain = by[("base", "plain")], by[("pureA", "plain")]
assert pure_plain.ci_low_substring > clean_plain.ci_high_substring, (
    "the ceiling arm did not transmit -- the corpus or the recipe is wrong, not the dose"
)
clean_moved = (
    clean_plain.ci_low_substring > base_plain.ci_high_substring
    or clean_plain.ci_high_substring < base_plain.ci_low_substring
)
print(f"pureA ceiling : {pure_plain.rate_substring:.4f}  (I7 measured 0.678 for pure jeqcho A)")
print(f"base          : {base_plain.rate_substring:.4f}")
print(f"clean vs base : {'MOVED -- report this' if clean_moved else 'indistinguishable, as expected'}")

In [ ]:
FRACTION = passing[0]
gate = {
    "fraction": FRACTION,
    "rule": RULE,
    "passing": passing,
    "variant": "plain",
    "metric": "rate_substring",
    "rates": {
        name: {
            "rate": by[(name, "plain")].rate_substring,
            "ci": [by[(name, "plain")].ci_low_substring, by[(name, "plain")].ci_high_substring],
        }
        for name in ("base",) + ARMS
    },
    "clean_moved_from_base": bool(clean_moved),
    "git_sha": config.git_sha(),
}
(RUN / "gate.json").write_text(json.dumps(gate, indent=2))
print(json.dumps(gate, indent=2))

## Gate result

_Fill in:_ chosen fraction **____**, P(cat) **____** vs clean **____**.
Everything downstream reads `RUN/gate.json`.

In [ ]:
print(f"wall clock: {(time.time() - T0) / 60:.1f} min")

### Attended time

_Fill in before committing:_ **__ min** attended.
Copy the wall clock above and this figure into `docs/compute_log.md`.